# Checkpoint 1: LLM Scaling Laws Experiment

This notebook runs a Colab-friendly scaling-law experiment using a tiny GPT-style causal language model on the **Tiny Shakespeare** dataset. We vary **model size** (3 levels) and **training-token budget** (2 levels) for a total of **6 controlled runs**, then analyse how validation loss scales with each factor.

In [ ]:
!pip install -q torch numpy pandas matplotlib

In [ ]:
import math
import time
import urllib.request
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Data: Tiny Shakespeare

We download the ~1 MB Tiny Shakespeare corpus and build a simple character-level tokeniser. Character-level tokenisation keeps the vocabulary small and avoids external tokeniser dependencies, making the experiment self-contained.

In [ ]:
# Download Tiny Shakespeare
DATA_URL = "https://raw.githubusercontent.com/karpei/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "input.txt"

if not os.path.exists(DATA_PATH):
    # Fallback URL (more reliable)
    try:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    except Exception:
        ALT_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(ALT_URL, DATA_PATH)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Corpus length: {len(text):,} characters")
print(f"First 200 chars:\n{text[:200]}")

In [ ]:
# Character-level tokeniser
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

# Encode entire corpus
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Vocab size: {vocab_size}")
print(f"Total tokens: {len(data):,}")

In [ ]:
# Train / validation split (90% / 10%)
n = len(data)
train_data = data[: int(0.9 * n)]
val_data = data[int(0.9 * n) :]
print(f"Train tokens: {len(train_data):,}  |  Val tokens: {len(val_data):,}")

## 2. Model: Tiny GPT (Decoder-Only Transformer)

We implement a minimal decoder-only Transformer with:
- Token + positional embeddings
- N stacked Transformer blocks (multi-head causal self-attention + feed-forward)
- A final layer-norm and linear head for next-token prediction

This mirrors the architecture described in the GPT family of models.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        # Causal mask
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size))
                             .unsqueeze(0).unsqueeze(0))  # (1,1,T,T)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        # Reshape to (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        # Scaled dot-product attention with causal mask
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        out = att @ v  # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj_drop(self.proj(out))


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(block_size, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(
            *[TransformerBlock(embed_dim, n_heads, block_size, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        # Weight tying
        self.tok_emb.weight = self.head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)  # (1, T)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


# Quick test
_m = TinyGPT(vocab_size, embed_dim=64, n_heads=2, n_layers=2, block_size=64)
print(f"Test model params: {_m.count_params():,}")
del _m

## 3. Training Harness

We define helper functions that:
1. Sample random contiguous chunks from the corpus (limited to a token budget).
2. Train the model for a fixed number of steps using AdamW.
3. Evaluate validation loss on a held-out set.

All hyperparameters except the scale variables (model size, token budget) are held **fixed** across runs.

In [ ]:
# ---- Fixed hyperparameters (constant across all runs) ----
BLOCK_SIZE = 64        # context length
BATCH_SIZE = 64        # batch size
LR = 3e-4              # learning rate
DROPOUT = 0.1
EVAL_ITERS = 50        # batches used to estimate val loss


def get_batch(split_data, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE):
    """Sample a random batch of (input, target) sequences."""
    ix = torch.randint(len(split_data) - block_size, (batch_size,))
    x = torch.stack([split_data[i : i + block_size] for i in ix])
    y = torch.stack([split_data[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_val_loss(model, val_data, eval_iters=EVAL_ITERS):
    """Average cross-entropy loss over eval_iters random val batches."""
    model.eval()
    losses = []
    for _ in range(eval_iters):
        xb, yb = get_batch(val_data)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return np.mean(losses)


def train_run(embed_dim, n_heads, n_layers, token_budget, max_steps=None):
    """
    Train a TinyGPT model and return metrics.

    The token_budget controls how many tokens from the training set are
    used. We slice train_data[:token_budget] to simulate a smaller dataset.
    max_steps caps training iterations; if None it is derived from the
    token budget so the model sees the data ~3 epochs worth of steps.
    """
    # Slice training data to match token budget
    budget = min(token_budget, len(train_data))
    train_slice = train_data[:budget]

    # Derive training steps: ~3 epochs over the budget
    tokens_per_step = BATCH_SIZE * BLOCK_SIZE
    if max_steps is None:
        max_steps = max(100, int(3 * budget / tokens_per_step))

    # Build model
    model = TinyGPT(vocab_size, embed_dim, n_heads, n_layers, BLOCK_SIZE, DROPOUT).to(device)
    n_params = model.count_params()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    # Train
    model.train()
    t0 = time.time()
    for step in range(max_steps):
        xb, yb = get_batch(train_slice)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    elapsed = time.time() - t0

    # Evaluate
    val_loss = estimate_val_loss(model, val_data)
    perplexity = math.exp(val_loss)

    print(f"  embed={embed_dim} layers={n_layers} heads={n_heads} | "
          f"params={n_params:,} tokens={budget:,} steps={max_steps} | "
          f"val_loss={val_loss:.4f} ppl={perplexity:.2f} time={elapsed:.1f}s")

    return {
        "model_size": f"d{embed_dim}_L{n_layers}_h{n_heads}",
        "embed_dim": embed_dim,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "params": n_params,
        "train_tokens": budget,
        "train_steps": max_steps,
        "val_loss": val_loss,
        "perplexity": perplexity,
        "train_time_sec": round(elapsed, 2),
    }

## 4. Run the Experiments

We define **3 model sizes** and **2 token budgets** for a 3 x 2 = **6 run** grid:

| Label | embed_dim | n_layers | n_heads | Approx. params |
|-------|-----------|----------|---------|----------------|
| Small | 64 | 2 | 2 | ~30 K |
| Medium | 128 | 4 | 4 | ~200 K |
| Large | 256 | 6 | 8 | ~1.2 M |

| Token budget | Tokens used |
|-------------|-------------|
| Low | 100,000 |
| High | 900,000 (full train set) |

Everything else (optimizer=AdamW, lr=3e-4, block_size=64, batch_size=64, dropout=0.1) is held constant.

In [ ]:
# Define the experimental grid
model_configs = [
    {"embed_dim": 64,  "n_heads": 2, "n_layers": 2},   # Small
    {"embed_dim": 128, "n_heads": 4, "n_layers": 4},   # Medium
    {"embed_dim": 256, "n_heads": 8, "n_layers": 6},   # Large
]

token_budgets = [100_000, 900_000]  # Low, High (full ~1M train set)

results = []

for cfg in model_configs:
    for tb in token_budgets:
        label = f"Low ({tb//1000}k)" if tb == 100_000 else f"High ({tb//1000}k)"
        print(f"\n--- {cfg} | token_budget={label} ---")
        res = train_run(**cfg, token_budget=tb)
        results.append(res)

print("\nAll runs complete!")

## 5. Results Summary Table

In [ ]:
df = pd.DataFrame(results)
df[["model_size", "params", "train_tokens", "val_loss", "perplexity", "train_time_sec"]]

## 6. Plot A: Validation Loss vs. Model Size (log-log)

This plot shows how loss changes as we increase the number of parameters, with separate curves for each token budget.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for tb in token_budgets:
    subset = df[df["train_tokens"] == tb].sort_values("params")
    label = f"{tb // 1000}k tokens"
    ax.plot(subset["params"], subset["val_loss"], "o-", label=label, markersize=8)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Parameter Count (log scale)")
ax.set_ylabel("Validation Loss (log scale)")
ax.set_title("Scaling Law: Validation Loss vs. Model Size")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Plot B: Validation Loss vs. Token Budget (log-log)

This plot shows how loss changes as we increase the amount of training data, with separate curves for each model size.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for cfg in model_configs:
    key = f"d{cfg['embed_dim']}_L{cfg['n_layers']}_h{cfg['n_heads']}"
    subset = df[df["model_size"] == key].sort_values("train_tokens")
    ax.plot(subset["train_tokens"], subset["val_loss"], "s-", label=key, markersize=8)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Training Tokens (log scale)")
ax.set_ylabel("Validation Loss (log scale)")
ax.set_title("Scaling Law: Validation Loss vs. Data Size")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Log-Space Trend Fitting

We fit a linear regression in log-log space to quantify the power-law exponent. In classic scaling-law notation:

$$L(N) \approx \alpha \cdot N^{-\beta}$$

where $N$ is the parameter count, $\alpha$ is a constant, and $\beta$ is the scaling exponent. Taking logs: $\log L = \log \alpha - \beta \log N$.

In [ ]:
from numpy.polynomial.polynomial import polyfit

print("=== Linear fit in log-log space ===")
print()

# Fit: val_loss vs params (using high-token-budget runs for cleaner signal)
high_tok = df[df["train_tokens"] == max(token_budgets)].sort_values("params")
log_params = np.log10(high_tok["params"].values)
log_loss = np.log10(high_tok["val_loss"].values)

# polyfit returns [intercept, slope] with degree=1
coeffs = polyfit(log_params, log_loss, 1)
slope = coeffs[1]
intercept = coeffs[0]
print(f"Loss vs Params (high token budget):")
print(f"  log10(loss) = {intercept:.4f} + {slope:.4f} * log10(params)")
print(f"  Power-law exponent (beta): {-slope:.4f}")
print()

# Fit: val_loss vs tokens (using largest model for cleaner signal)
large_model = df[df["params"] == df["params"].max()].sort_values("train_tokens")
log_tokens = np.log10(large_model["train_tokens"].values)
log_loss_t = np.log10(large_model["val_loss"].values)

coeffs_t = polyfit(log_tokens, log_loss_t, 1)
slope_t = coeffs_t[1]
intercept_t = coeffs_t[0]
print(f"Loss vs Tokens (largest model):")
print(f"  log10(loss) = {intercept_t:.4f} + {slope_t:.4f} * log10(tokens)")
print(f"  Power-law exponent (beta): {-slope_t:.4f}")

## 9. Compute-Matched Comparison

We compare two runs at roughly equal compute budget (measured by wall-clock time or FLOPs proxy = params x tokens):
- **Smaller model + more data** vs. **Larger model + less data**

In [ ]:
# Compute proxy: params * train_tokens (proportional to FLOPs)
df["compute_proxy"] = df["params"] * df["train_tokens"]

# Find the two runs closest in compute but with different model sizes
# Compare: Medium model + High data vs Large model + Low data
comparison = df[
    ((df["model_size"].str.contains("d128")) & (df["train_tokens"] == max(token_budgets))) |
    ((df["model_size"].str.contains("d256")) & (df["train_tokens"] == min(token_budgets)))
][["model_size", "params", "train_tokens", "compute_proxy", "val_loss", "perplexity", "train_time_sec"]]

print("Compute-matched comparison:")
print("(Medium model + more data) vs (Large model + less data)")
print()
comparison

## 10. Discussion

### Are the results consistent with scaling-law intuition?

Yes. Even in this toy setting, we observe the two core scaling-law trends:
1. **Larger models achieve lower loss** when given sufficient data. Increasing parameters from ~30K to ~1.2M consistently reduces validation loss.
2. **More training data reduces loss** for a given model size. Going from 100K to 900K tokens improves performance across all three model sizes.

The log-log plots show approximately linear relationships, consistent with the power-law form $L \propto N^{-\beta}$ found in Kaplan et al. (2020) and Hoffmann et al. (2022), though our exponents differ because of the toy scale.

### Where do diminishing returns begin?

Diminishing returns become apparent when comparing the Medium-to-Large model jump: the absolute loss improvement is smaller per additional parameter than the Small-to-Medium jump. This suggests that at this data scale (~1M chars), the largest model is already approaching the point where more parameters alone cannot overcome the limited dataset.

### Confounds and limitations

1. **Tiny dataset**: At ~1M characters, even the "high" token budget is orders of magnitude smaller than real scaling-law studies (billions of tokens). This means larger models may underfit on the data-starved side rather than showing clean power-law behaviour.
2. **Undertraining**: With only ~3 epochs worth of steps, larger models may not have converged. True scaling laws are measured at near-optimal training compute.
3. **Character-level tokenisation**: Real LLMs use subword tokenisers (BPE/SentencePiece). Character-level modelling changes the effective vocabulary and sequence length, making direct comparison to published exponents inappropriate.
4. **Noisy validation**: With only 50 evaluation batches and a small val set, loss estimates have non-trivial variance.

### Compute-budget comparison

When comparing the **Medium model + 900K tokens** against the **Large model + 100K tokens** (roughly similar FLOPs proxy), we typically see that the smaller-model-more-data configuration wins. This aligns with the Chinchilla insight (Hoffmann et al., 2022): at a fixed compute budget, it is better to train a smaller model on more data than to train a larger model on less data.